## 🚀 TFB Prediction Tutorial 3/4: Model Training - Teaching the Model to Predict

Welcome to the third tutorial in our series. So far, we have:
1.  Prepared our dataset and dataloaders (`01_data_preparation.ipynb`).
2.  Selected and initialized the correct model architecture (`02_model_initialization.ipynb`).

Now, we have all the ingredients ready. In this tutorial, we will combine them to **train** the model. Training is the process of teaching the model to make accurate predictions by showing it examples from our dataset and adjusting its internal parameters.

This tutorial will cover:
1.  **The Concept of Supervised Training**: What it means to train a model and what components are required.
2.  **Trainers in `OmniGenBench`**: An overview of the different training engines available in the framework.
3.  **The Training Process in Action**: A step-by-step guide to configuring and launching the training job.
4.  **Evaluation and Results**: How we measure the model's performance and what artifacts are produced.

By the end of this tutorial, you will have a fine-tuned model saved on your disk, ready for inference.

### 1. The Concept of Supervised Training

At its heart, **supervised training** is like teaching a student with a textbook and an answer key. We show the model an input (a DNA sequence), let it make a prediction, and then compare its prediction to the correct answer (the labels). The difference between the prediction and the answer is quantified by a **loss function**. The goal is to minimize this loss.

This process is iterative and happens in a **training loop**, which consists of four main steps:
1.  **Forward Pass**: The input data is passed through the model to get a prediction.
2.  **Loss Calculation**: The model's prediction is compared to the ground-truth labels using a loss function (e.g., Binary Cross-Entropy for our task).
3.  **Backward Pass (Backpropagation)**: The loss is used to calculate the gradient for each of the model's parameters. The gradient tells us how to adjust each parameter to reduce the loss.
4.  **Optimizer Step**: An **optimizer** (like AdamW) updates the model's parameters based on the calculated gradients.

To manage this entire process, we need a few key components:
-   A **Model** to be trained.
-   **DataLoaders** to supply batches of data for training, validation, and testing.
-   A **Loss Function** to measure prediction error.
-   An **Optimizer** to update the model's weights.
-   An **Evaluation Metric** (e.g., ROC-AUC) to assess performance on a validation set.

A **Trainer** is an object that encapsulates this entire training loop, abstracting away the boilerplate code and providing a clean interface to manage training, evaluation, and logging.

### 2. Trainers in `OmniGenBench`

`OmniGenBench` provides a flexible system for training by offering several "Trainer" classes, each suited for different needs. All trainers handle the core training loop, but they differ in their features and complexity.

| Trainer Class         | Key Feature                               | When to Use                                                                                             |
| --------------------- | ----------------------------------------- | ------------------------------------------------------------------------------------------------------- |
| `Trainer`             | Native PyTorch Implementation             | For simple, single-GPU training. It's lightweight and gives you a clear, direct PyTorch experience.      |
| `AccelerateTrainer`   | Distributed Training via `accelerate`     | **Recommended for most users.** Easily scales from single-GPU to multi-GPU or multi-node setups with minimal code changes. |
| `HFTrainer`           | Integration with Hugging Face `Trainer`   | If you are already heavily invested in the Hugging Face ecosystem and prefer its `TrainingArguments` setup. |

For this tutorial, we will use the **`AccelerateTrainer`**. It represents the best balance of power and ease of use, making it simple to run our experiment on a single GPU today and scale it up to a powerful cluster tomorrow if needed. The `accelerate` library by Hugging Face handles all the complexities of distributed training behind the scenes.

### 3. The Training Process in Action

Let's put everything together and train our model. The process involves three main code blocks:
1.  **Re-establishing the context**: We'll quickly re-run the code from the previous tutorials to get our `dataloaders`, `model`, and `tokenizer`.
2.  **Defining the evaluation metric**: We'll set up the function to compute ROC-AUC during validation.
3.  **Configuring and running the `AccelerateTrainer`**: This is where we define all training parameters and launch the job.

#### 3.1. Setup: Data and Model Initialization

First, let's import everything we need and run the setup code from the previous tutorials. This ensures we have all the necessary objects in our environment.

In [ ]:
# Import libraries
import os
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

from omnigenbench import (
    OmniTokenizer,
    OmniModelForMultiLabelSequenceClassification,
    OmniDatasetForSequenceClassification,
    ClassificationMetric,
    AccelerateTrainer
)
from autocuda import auto_cuda

print("✅ Libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Training Configuration
class TFBTrainingConfig:
    MODEL_NAME = "yangheng/OmniGenome-52M"
    DATASET_NAME = "yangheng/deepsea_tfb_prediction"
    
    # Model parameters
    NUM_LABELS = 919  # Full TF set (use 10 for testing)
    MAX_LENGTH = 512
    
    # Training parameters
    BATCH_SIZE = 64
    LEARNING_RATE = 2e-5
    EPOCHS = 3
    PATIENCE = 3
    
    # Output
    OUTPUT_DIR = "./checkpoints/deepsea_finetune"
    SEED = 42
    
    # Device
    DEVICE = auto_cuda()

config = TFBTrainingConfig()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print("⚙️ TFB Training Configuration:")
print(f"  🧬 Model: {config.MODEL_NAME}")
print(f"  📊 Dataset: {config.DATASET_NAME}")
print(f"  🎯 Labels: {config.NUM_LABELS}")
print(f"  📱 Device: {config.DEVICE}")
print(f"  📁 Output: {config.OUTPUT_DIR}")
print("✅ Configuration ready!")

In [ ]:
# Load tokenizer and prepare datasets
print("🔄 Loading tokenizer...")
tokenizer = OmniTokenizer.from_pretrained(config.MODEL_NAME, trust_remote_code=True)

print("📊 Loading DeepSEA TFB dataset...")
datasets = OmniDatasetForSequenceClassification.from_huggingface(
    dataset_name=config.DATASET_NAME,
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH,
    cache_dir="deepsea_tfb_cache"
)

# Create DataLoaders
train_loader = datasets['train'].get_dataloader(
    batch_size=config.BATCH_SIZE, 
    shuffle=True
)
valid_loader = datasets['valid'].get_dataloader(
    batch_size=config.BATCH_SIZE, 
    shuffle=False
)

print(f"📋 Dataset prepared:")
print(f"  📈 Training samples: {len(datasets['train'])}")
print(f"  🔍 Validation samples: {len(datasets['valid'])}")
print(f"  📦 Batch size: {config.BATCH_SIZE}")

# Load model
print("🔄 Loading model...")
model = OmniModelForMultiLabelSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=config.NUM_LABELS,
    trust_remote_code=True
)
model.to(config.DEVICE)
print(f"✅ Model loaded with {config.NUM_LABELS} labels on {config.DEVICE}")

print("✅ Training setup complete!")

In [ ]:
# --- Model Initialization (from Tutorial 02) ---
tokenizer = AutoTokenizer.from_pretrained(config["model_name_or_path"])
model = OmniModelForMultiLabelSequenceClassification.from_pretrained(
    config["model_name_or_path"],
    num_labels=config["num_labels"],
    trust_remote_code=True
)
model.to(config["device"])

train_dataloader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False)

print("Setup complete. DataLoaders, Model, and Tokenizer are ready.")

#### 3.2. Defining the Evaluation Metric

During training, we need to monitor how well our model is performing on data it hasn't seen before (the validation set). This helps us avoid overfitting and tells us when to stop training. For multi-label classification tasks, **Area Under the Receiver Operating Characteristic Curve (ROC-AUC)** is a standard and robust metric.

We will define a `compute_metrics` function that takes the model's predictions and the true labels and returns the average ROC-AUC score across all labels. The `AccelerateTrainer` will automatically call this function at the end of each epoch.

In [ ]:
# In pratice, we can import metrics from omnigenbench
# compute_metrics = roc_auc_score
# or 
compute_metrics = ClassificationMetric().roc_auc_sccore

# or you can define the metric calculation method
def compute_metrics(all_preds, all_labels):
    """
    Compute ROC-AUC score.
    """
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    # Apply sigmoid to get probabilities
    probs = 1 / (1 + np.exp(-all_preds))
    
    # Calculate ROC-AUC for each label and average
    auc_scores = []
    for i in range(all_labels.shape[1]):
        try:
            auc = roc_auc_score(all_labels[:, i], probs[:, i])
            auc_scores.append(auc)
        except ValueError:
            # This can happen if a label has only one class in the batch
            pass
            
    mean_auc = np.mean(auc_scores)
    return {"roc_auc": mean_auc} 

#### 3.3. Configuring and Running the `AccelerateTrainer`

This is the final and most important step. We will instantiate the `AccelerateTrainer` and provide it with all the necessary components and hyperparameters.

Key parameters for the trainer include:
-   `model`, `train_dataloader`, `valid_dataloader`: The core components for training.
-   `epochs`: The total number of times to iterate over the entire training dataset.
-   `optimizer_class`: The optimizer to use (we'll use `torch.optim.AdamW`, a standard choice for transformer models).
-   `lr`: The learning rate, which controls the step size of the optimizer.
-   `compute_metrics`: Our evaluation function.
-   `output_dir`: The directory where checkpoints and results will be saved.
-   `early_stopping_patience`: A crucial parameter for preventing overfitting. The training will stop if the validation metric (`roc_auc`) does not improve for this many epochs.
-   `monitor_metric`: The metric to watch for early stopping and for saving the best model checkpoint.

Let's create the trainer and start the training process by calling the `.train()` method.

In [ ]:
# Initialize the trainer
trainer = AccelerateTrainer(
    model=model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    epochs=config["epochs"],
    optimizer=torch.optim.AdamW,
    lr=config["lr"],
    compute_metrics=compute_metrics,
    output_dir=config["output_dir"],
    patience=config["patience"],
)

# Start training
print("Starting training...")
trainer.train()
print("Training finished.")

### 4. Evaluation and Results

After the `trainer.train()` call completes, the training process is finished. So, what did we get?

The `AccelerateTrainer` automatically handles several important things:
1.  **Logging**: During training, it prints the progress, including the training loss and the validation metrics for each epoch. You should see the `roc_auc` score improving over time.
2.  **Model Checkpointing**: The trainer saves the model weights of the best-performing epoch (based on `monitor_metric`) to the specified `output_dir`. Inside the `./checkpoints/deepsea_finetune` folder, you will find a file named `best_model.pth`. This is your fine-tuned model, ready for use.
3.  **Results Summary**: A JSON file named `all_results.json` is also saved in the output directory, containing the performance metrics from each epoch. This is useful for tracking the training process and for later analysis.

Let's take a look at the results file:

In [ ]:
# Load and print the results
results_file = os.path.join(config["output_dir"], "all_results.json")
if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)
    print("Training results:")
    print(json.dumps(results, indent=2))
else:
    print("Results file not found.")

# The best model is saved at:
best_model_path = os.path.join(config["output_dir"], "best_model.pth")
print(f"\nBest model saved at: {best_model_path}")

### Summary and Next Steps

In this tutorial, we have successfully fine-tuned our Genomic Foundation Model for the task of TFB prediction.

We have learned about:
-   The core concepts of supervised training.
-   The different trainers available in `OmniGenBench` and why `AccelerateTrainer` is a good choice.
-   How to configure and launch a training job, including setting up an evaluation metric and early stopping.
-   The outputs of the training process: a trained model checkpoint (`best_model.pth`) and a results summary (`all_results.json`).

We now have a model that has learned to identify transcription factor binding sites from raw DNA sequences. But a trained model is only useful if we can use it to make predictions on new, unseen data.

In the final tutorial of this series, **[4/4: Model Inference](./04_model_inference.ipynb)**, we will learn how to load our saved model and use it to perform inference, evaluate its performance on the test set, and interpret the results.